## Deep Research — Production-Annotated Version

This is the original lab notebook, refactored to call into a proper `deep_research/` package
(see the accompanying project files) instead of defining everything inline in cells.

**What changed and why is called out in the markdown cell above each step.** The actual
implementation now lives in reusable, testable modules:

```
deep_research/
  config.py          # env loading, logging, startup validation
  schemas.py          # WebSearchItem, WebSearchPlan, ReportData
  search_tool.py       # web_search tool (DDGS + retries)
  email_tool.py        # send_email_tool (guarded messenger import)
  llm_clients.py        # provider-selectable model construction
  agents_setup.py       # builds the 4 Agent objects
  research_manager.py    # orchestration: plan -> search -> write -> save -> email
```

See `README.md` in the project for the full list of production changes and setup instructions.


### Step 1 — Configuration

**Notebook originally did:** `load_dotenv(override=True)` then read each API key into its own
module-level variable, with all three provider clients built unconditionally even though only
one (`google_model`) was ever used.

**Production change:** configuration now lives in `deep_research/config.py`, is validated once
at startup (`validate_config()`), and only the client for the *selected* `MODEL_PROVIDER` gets
built. This means a missing `GOOGLE_API_KEY` fails immediately with a clear message, instead of
surfacing as a confusing 401 three agent calls deep — and you're not required to hold Groq/
OpenRouter credentials you never use.


In [ ]:
from deep_research.config import validate_config, logger

# Fails fast, with a clear message, if the selected provider's API key is missing.
validate_config()


### Step 2 — The agents

**Notebook originally did:** defined 4 agents inline across many cells, with instructions built
as f-strings referencing module-level globals (`HOW_MANY_SEARCHES`) at import time.

**Production change:** all four agents (Research, Planner, Writer, Email) are built by
`deep_research.agents_setup.build_agents()`. Instructions are plain module-level string
constants — easy to unit test, easy to diff in code review, and not dependent on cell
execution order.


In [ ]:
from deep_research.agents_setup import build_agents

agents = build_agents()
agents


### Step 3 — Run the pipeline

**Notebook originally did:** `run_searches`, `perform_search`, `write_report`, and
`send_report_email` were separate top-level functions, with retry logic only on the *planner*
call, and a bare `asyncio.gather` that would abort the whole run if any single search failed.

**Production change:** `deep_research.research_manager.run_deep_research(query)` wraps the
whole flow:
- every agent call (not just the planner) retries with backoff,
- `asyncio.gather(..., return_exceptions=True)` means one failed search is dropped and logged,
  not fatal,
- the report is written to `reports/<timestamp>_<query>.md` **before** the email step, so a
  broken email integration never loses a finished report,
- email failures are logged, not raised — by that point the report already exists on disk.


In [ ]:
from deep_research import run_deep_research

query = "Most popular AI Agent frameworks in 2026"
report = await run_deep_research(query)

print(report.short_summary)


### Step 4 — Inspect the result

Same `ReportData` schema as the original notebook (`short_summary`, `markdown_report`,
`follow_up_questions`) — no changes needed downstream if you already consume this shape.


In [ ]:
from IPython.display import Markdown, display

display(Markdown(report.markdown_report))
print("\nFollow-up questions:")
for q in report.follow_up_questions:
    print(f"- {q}")


### As always, take a look at the trace

https://platform.openai.com/traces

**Production note:** the `trace(...)` context manager (used inside `run_deep_research`) reports
to OpenAI's traces dashboard. Confirm that's acceptable for your data before shipping this to
production, or swap in a custom tracing processor if you need results to stay in-house.
